# VCM integration and subclustering

This notebook integrates VCM datasets and performs downstream subclustering. This GitHub-oriented copy has notebook outputs removed, imports consolidated, and explanatory section headings added while preserving the original analytical workflow.

> **Before public release:** verify that remaining paths, sample identifiers, and metadata fields are appropriate to publish, and document input data and package versions in the repository README.


## Setup

Centralized imports used by the analysis.


In [ ]:
import h5py
import warnings
import os
import spatialdata_plot
import logging
from pathlib import Path


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sio


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:

# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
!pwd


In [ ]:
# Set `sc.settings.figdir` for the following analysis.
sc.settings.figdir = "Thesis_figures/snRNAseq/VCMs"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Compute and store `out_dir`.
out_dir = Path("Thesis_figures/snRNAseq/VCMs")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


## Load data

Load expression data and associated metadata.


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("Allwithall4plex_filtered_integratedbysample_annotated_withsubtypes_withCas9_ordered.h5ad")


## Subset ventricular cardiomyocytes


In [ ]:
# Make an independent copy of the selected data.
VCMs = adata[adata.obs['celltype'] == 'VCMs'].copy()


In [ ]:
# Load the processed AnnData object.
VCMs = sc.read_h5ad("VCMs.h5ad")
# Make an independent copy of the selected data.
VCMs.X = VCMs.layers["counts"].copy()


## Normalization and feature selection

Normalize expression values and identify informative features.


In [ ]:
# normalize/log for PCA
sc.pp.normalize_total(VCMs, target_sum=1e4)
# Log-transform the normalized expression values.
sc.pp.log1p(VCMs)


In [ ]:
# Identify highly variable genes.
sc.pp.highly_variable_genes(VCMs, n_top_genes=2000, batch_key="sample")


In [ ]:
# Compute and store `VCMs.layers['scaled']`.
VCMs.layers["scaled"] = VCMs.X.toarray()
# Run `sc.pp.regress_out` for this analysis step.
sc.pp.regress_out(VCMs, ["total_counts", "pct_counts_mt"], layer="scaled")
# Scale gene expression for downstream dimensionality reduction.
sc.pp.scale(VCMs, max_value=10, layer="scaled")


## Dimensionality reduction

Build reduced-dimensional representations and a neighborhood graph.


In [ ]:
# Compute principal components.
sc.tl.pca(VCMs, layer = "scaled")


In [ ]:
# Run `sc.pl.pca_variance_ratio` for this analysis step.
sc.pl.pca_variance_ratio(VCMs, n_pcs=50, log=True)


## Dataset integration

Combine datasets and address sample/batch structure before downstream analysis.


In [ ]:
# re-run Harmony (key is your batch column, e.g. "batch" or "sample")
sc.external.pp.harmony_integrate(VCMs, key=["sample"])


In [ ]:
# Build the cell-neighbour graph in the selected representation.
sc.pp.neighbors(VCMs, use_rep="X_pca_harmony", n_pcs = 20)
# Compute the UMAP embedding.
sc.tl.umap(VCMs)


## Clustering and subclustering

Resolve transcriptionally related populations at the desired granularity.


In [ ]:
# Cluster cells using the Leiden algorithm.
sc.tl.leiden(VCMs, resolution=0.4, key_added="leid04")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = ['name','sample_type','plex','leid02', 'leid03', 'leid04', 'doublet_score'], size = 15)


## Export results

Save processed objects, tables, and/or figures for downstream use.


In [ ]:
# Save the processed AnnData object to disk.
VCMs.write_h5ad("VCMs_harmonybysample_regressedout.h5ad")


## Marker-gene analysis

Inspect cluster-associated genes to support interpretation of the resulting populations.


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(VCMs, groupby = "leid02", method = "wilcoxon")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "leid02", standard_scale="var", n_genes=10)


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(VCMs, groupby = "leid03", method = "wilcoxon")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "leid03", standard_scale="var", n_genes=10)


In [ ]:
# Map Leiden cluster IDs to cell type labels
cluster_map = {
    "0": "CM2", #intermediate
    "1": "CM1",
    "2": "CM3 (stressed)",
    "3": "doublets",
    "4": "CM5",
    "5": "doublets",
    "6": "CM4 (Ifgga4 high)",
    "7": "Myeloid"
    # ... extend as needed
}

# Compute and store `VCMs.obs['celltype']`.
VCMs.obs["celltype"] = VCMs.obs["leid03"].map(cluster_map)


In [ ]:
# Save the processed AnnData object to disk.
VCMs.write_h5ad("VCMs_harmonybysample_regressedout_annotated.h5ad")


In [ ]:
# Load the processed AnnData object.
VCMs = sc.read_h5ad("VCMs_harmonybysample_regressedout_annotated.h5ad")


In [ ]:
# Make an independent copy of the selected data.
VCMs = VCMs[VCMs.obs["celltype"].isin(["CM1", "CM2", "CM3 (stressed)", "CM4 (Ifgga4 high)", "CM5"])].copy()


In [ ]:
# Define the values used for `order`.
order = ['CM1', 'CM2', 'CM3 (stressed)', 'CM4 (Ifgga4 high)', 'CM5']


In [ ]:
# Compute and store `VCMs.obs['celltype']`.
VCMs.obs["celltype"] = pd.Categorical(VCMs.obs["celltype"], ordered = True, categories=order)


In [ ]:
# Define the values used for `set1_92`.
set1_92 = [
    "#6C5C8D", "#88CCEE", "#97B1AB", "#CC6677", "#117733", "#999933",
    "#AA4499", "#7BB4C3", "#FF987B","#44AA99"

]


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi_save = 300, figsize = (6, 6))
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = 'celltype', palette=set1_92, size = 13, save = "VCMs_umap_harmonybysample.pdf")


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi_save = 300, figsize = (6, 6))
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = 'sample_type', palette=["#6C5C8D", "#117733", "#97B1AB"] , size = 17, save = "VCMs_bysample_umap_harmonybysample.pdf")


In [ ]:
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi_save = 300, figsize = (6, 6))
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = 'plex', palette=["#6C5C8D", "#117733", "#97B1AB"] , size = 14, save = "VCMs_byplex_umap_harmonybysample.pdf")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = ["F830016B08Rik", "C_terminal_Cas9", "Abe8e_with_N_terminal_Cas9"], size = 17, save = "featuremap_Frik_Cas92_VCMs_harmonybysample.pdf")


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(VCMs, color = ["Ankrd1", "Xirp2", "Nppa"], size = 17, save = "featuremap_stress_VCMs_harmonybysample.pdf")


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(VCMs, groupby = "celltype", method = "wilcoxon")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, categories_order=order, groupby = "celltype", dendrogram=False, standard_scale="var", n_genes=5, save = "DotplotCMs_bysample.pdf")


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(VCMs, groupby = "Cas9_status", method = "wilcoxon")


In [ ]:
# Plot the distribution of the selected measurement across groups.
sc.pl.violin(VCMs, groupby = "Cas9_status", keys=["doublet_score", "total_counts"], jitter=0, palette=["#6C5C8D", "#88CCEE", "#97B1AB", "#117733"])


In [ ]:
# Set `key` for the following analysis.
key = "doublet_score"
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300, dpi_save = 300, transparent = True, figsize=(5,5))
# Compute and store `ax`.
ax = sc.pl.violin(
    VCMs,
    [key],
    jitter=0,
    palette=["#6C5C8D", "#88CCEE", "#97B1AB", "#117733"],
    groupby="Cas9_status",
    show=False,
    rotation = 45
)

# Rasterize scatter-like collections (usually the jitter points)
for coll in ax.collections:
    coll.set_rasterized(True)

# order of groups as shown in the plot
groups = [t.get_text() for t in ax.get_xticklabels()]

# mean per group, aligned to that order
means = adata.obs.groupby("Cas9_status")[key].mean().reindex(groups)

# draw one mean line per group (a short horizontal segment)
x = np.arange(len(groups))
# Run `ax.hlines` for this analysis step.
ax.hlines(means.values, x - 0.35, x + 0.35, linestyles="--", linewidth=1.5, colors="black")
# Set `fig` for the following analysis.
fig = ax.figure
# Save the completed figure to disk.
fig.savefig(out_dir / "doubletscorebyCas9_violin_withmean_filtered.pdf", bbox_inches="tight", dpi=300, transparent=True)
# Save the completed figure to disk.
fig.savefig(out_dir / "doubletscorebyCas9_violin_withmean_filtered.png", bbox_inches="tight", dpi=300, transparent=True)

# Run `plt.show` for this analysis step.
plt.show()


In [ ]:
# Set `key` for the following analysis.
key = "total_counts"
# Run `sc.set_figure_params` for this analysis step.
sc.set_figure_params(dpi = 300, dpi_save = 300, transparent = True, figsize=(5,5))
# Compute and store `ax`.
ax = sc.pl.violin(
    VCMs,
    [key],
    jitter=0,
    palette=["#6C5C8D", "#88CCEE", "#97B1AB", "#117733"],
    groupby="Cas9_status",
    show=False,
    rotation = 45
)

# Rasterize scatter-like collections (usually the jitter points)
for coll in ax.collections:
    coll.set_rasterized(True)

# order of groups as shown in the plot
groups = [t.get_text() for t in ax.get_xticklabels()]

# mean per group, aligned to that order
means = adata.obs.groupby("Cas9_status")[key].mean().reindex(groups)

# draw one mean line per group (a short horizontal segment)
x = np.arange(len(groups))
# Run `ax.hlines` for this analysis step.
ax.hlines(means.values, x - 0.35, x + 0.35, linestyles="--", linewidth=1.5, colors="black")
# Set `fig` for the following analysis.
fig = ax.figure
# Save the completed figure to disk.
fig.savefig(out_dir / "totalcounts_byCas9_violin_withmean_filtered.pdf", bbox_inches="tight", dpi=300, transparent=True)
# Save the completed figure to disk.
fig.savefig(out_dir / "totalcounts_byCas9_violin_withmean_filtered.png", bbox_inches="tight", dpi=300, transparent=True)

# Run `plt.show` for this analysis step.
plt.show()


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "Cas9_status", dendrogram=False, standard_scale="var", n_genes=7, save = "DotplotCMs_byCAs9status_integratedbysam.pdf")


In [ ]:
import scanpy as sc

# Define the values used for `exclude_genes`.
exclude_genes = ["Abe8e_with_N_terminal_Cas9", "C_terminal_Cas9"]

# get groups from your grouping column
groups = VCMs.obs["Cas9_status"].cat.categories.tolist()

# build one filtered gene list per group
filtered_genes = {}

# Repeat the following operation for each item in the selected collection.
for group in groups:
    df = sc.get.rank_genes_groups_df(VCMs, group=group)
    genes = [g for g in df["names"] if g not in exclude_genes]
    filtered_genes[group] = genes[:7]   # keep top 7 after filtering

# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(
    VCMs,
    groupby="Cas9_status",
    var_names=filtered_genes,
    dendrogram=False,
    standard_scale="var",
    save="DotplotCMs_byCas9status_integratedbysam_filterednocas9.pdf"
)


In [ ]:
# Make an independent copy of the selected data.
BE = VCMs[VCMs.obs['sample_type']=='BE'].copy()


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(BE, groupby = "Cas9_status", method = "wilcoxon")


In [ ]:
# Define the values used for `exclude_genes`.
exclude_genes = ["Abe8e_with_N_terminal_Cas9", "C_terminal_Cas9"]

# get groups from your grouping column
groups = BE.obs["Cas9_status"].cat.categories.tolist()

# build one filtered gene list per group
filtered_genes = {}

# Repeat the following operation for each item in the selected collection.
for group in groups:
    df = sc.get.rank_genes_groups_df(BE, group=group)
    genes = [g for g in df["names"] if g not in exclude_genes]
    filtered_genes[group] = genes[:7]   # keep top 7 after filtering

# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(
    BE,
    groupby="Cas9_status",
    var_names=filtered_genes,
    dendrogram=False,
    standard_scale="var",
    save="DotplotCMs_byCas9status_integratedbysam_filterednocas9_onlyBEsamples.pdf"
)


In [ ]:
# make a combined DE group
VCMs.obs["Cas9_sample_group"] = (
    VCMs.obs["sample_type"].astype(str) + "__" +
    VCMs.obs["Cas9_status"].astype(str)
)

# Inspect the number of observations in each category.
VCMs.obs["Cas9_sample_group"].value_counts()


In [ ]:
# remove all WT or R636Q groups that are NOT no_Cas9
to_remove = [
    g for g in VCMs.obs["Cas9_sample_group"].unique()
    if (g.startswith("WT__") or g.startswith("R636Q__")) and ("no_Cas9" not in g)
]

# Run `print` for this analysis step.
print("Removing:", to_remove)

# Make an independent copy of the selected data.
VCMs = VCMs[~VCMs.obs["Cas9_sample_group"].isin(to_remove)].copy()

# optional: clean up unused categories
#VCMs.obs["Cas9_sample_group"] = VCMs.obs["Cas9_sample_group"].cat.remove_unused_categories()

# check
VCMs.obs["Cas9_sample_group"].value_counts()


In [ ]:
# Identify genes associated with the selected clusters/groups.
sc.tl.rank_genes_groups(VCMs, groupby = "Cas9_sample_group", method = "wilcoxon")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "Cas9_sample_group", dendrogram=False, standard_scale="var", n_genes=5, save = "Dotplot_Cas9andsample.pdf")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "Cas9_sample_group", dendrogram=False, standard_scale="var", n_genes=10)


In [ ]:
# Define the values used for `exclude_genes`.
exclude_genes = ["Abe8e_with_N_terminal_Cas9", "C_terminal_Cas9"]

# get groups from your grouping column
groups = VCMs.obs["Cas9_sample_group"].cat.categories.tolist()

# build one filtered gene list per group
filtered_genes = {}

# Repeat the following operation for each item in the selected collection.
for group in groups:
    df = sc.get.rank_genes_groups_df(VCMs, group=group)
    genes = [g for g in df["names"] if g not in exclude_genes]
    filtered_genes[group] = genes[:6]   # keep top 7 after filtering
    
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = "Cas9_sample_group", 
                                var_names = filtered_genes,
                                dendrogram=False, standard_scale="var", save = "Dotplot_Cas9andsample_nocas9genees.pdf")


In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# settings
# ----------------------------
groupby = "Cas9_sample_group"

# Define the values used for `be_groups`.
be_groups = [
    "BE__Cas9_full",
    "BE__NtermCas9",
    "BE__CtermCas9",
    "BE__no_Cas9",
]

# Set `wt_ref` for the following analysis.
wt_ref = "WT__no_Cas9"
# Set `rq_ref` for the following analysis.
rq_ref = "R636Q__no_Cas9"

# optional prettier labels for the gene blocks on top
pretty_names = {
    "BE__Cas9_full": "BE Cas9 full",
    "BE__NtermCas9": "BE Cas9 Nterm",
    "BE__CtermCas9": "BE Cas9 Cterm",
    "BE__no_Cas9": "BE no Cas9",
}


# Define `make_be_vs_reference_matrixplot()` for reuse in the analysis below.
def make_be_vs_reference_matrixplot(
    adata,
    reference_group,
    target_groups,
    groupby="Cas9_sample_group",
    n_top=5,
    p_adj_cutoff=0.05,
    log2fc_min=0.25,
    use_raw=False,
    standard_scale="var",
    save=None,
):
    """
    For each target group:
      - run pairwise DE target vs reference
      - keep top significant up genes
    Then make one combined matrixplot and place a star in the cell
    corresponding to the target group for each selected gene.
    """

    de_tables = {}
    gene_blocks = {}
    column_targets = []   # one entry per plotted gene column, to know where to place the star

    # ----------------------------
    # pairwise DE per BE subgroup
    # ----------------------------
    for target in target_groups:
        sub = adata[adata.obs[groupby].isin([target, reference_group])].copy()

        key = f"DE__{target}__vs__{reference_group}"

        sc.tl.rank_genes_groups(
            sub,
            groupby=groupby,
            groups=[target],
            reference=reference_group,
            method="wilcoxon",
            pts=True,
            tie_correct=True,
            use_raw=use_raw,
            key_added=key,
        )

        df = sc.get.rank_genes_groups_df(
            sub,
            group=target,
            key=key,
            pval_cutoff=p_adj_cutoff,
            log2fc_min=log2fc_min,
        )

        # keep genes higher in BE target than in reference
        df = df.sort_values(
            ["logfoldchanges", "pvals_adj"],
            ascending=[False, True]
        ).drop_duplicates(subset="names")

        de_tables[target] = df.copy()

        top_genes = df["names"].head(n_top).tolist()
        gene_blocks[pretty_names.get(target, target)] = top_genes
        column_targets.extend([target] * len(top_genes))

    # remove empty blocks if a group had no sig genes
    gene_blocks = {k: v for k, v in gene_blocks.items() if len(v) > 0}
    column_targets = [
        t for block_label, genes in gene_blocks.items()
        for t in [next(k for k, v in pretty_names.items() if v == block_label)] * len(genes)
    ]

    # ----------------------------
    # plot only relevant groups
    # ----------------------------
    plot_order = [reference_group] + target_groups
    plot_adata = adata[adata.obs[groupby].isin(plot_order)].copy()

    total_genes = sum(len(v) for v in gene_blocks.values())
    fig_w = max(8, total_genes * 0.45)

    mp = sc.pl.matrixplot(
        plot_adata,
        var_names=gene_blocks,
        groupby=groupby,
        categories_order=plot_order,
        use_raw=use_raw,
        standard_scale=standard_scale,
        dendrogram=False,
        figsize=(fig_w, 4),
        return_fig=True,
    )

    # render and get axes
    mp.make_figure()
    axes = mp.get_axes()
    ax = axes["mainplot_ax"]

    # row positions from displayed y tick labels
    y_labels = [t.get_text() for t in ax.get_yticklabels()]
    y_ticks = ax.get_yticks()
    y_map = dict(zip(y_labels, y_ticks))

    # x positions correspond to the plotted columns in order
    x_ticks = ax.get_xticks()

    # place one star in the row of the target group for each selected gene
    for x, target in zip(x_ticks, column_targets):
        if target in y_map:
            ax.text(
                x, y_map[target], "★",
                ha="center", va="center",
                fontsize=8, color="black", fontweight="bold"
            )

    ax.set_title(f"Top BE subgroup markers vs {reference_group}", pad=20)

    if save is not None:
        mp.savefig(save, bbox_inches="tight")

    plt.show()

    return de_tables, gene_blocks


In [ ]:
# Compute and store `(de_vs_wt, genes_vs_wt)`.
de_vs_wt, genes_vs_wt = make_be_vs_reference_matrixplot(
    VCMs,
    reference_group="WT__no_Cas9",
    target_groups=be_groups,
    groupby="Cas9_sample_group",
    n_top=10,
    p_adj_cutoff=0.05,
    log2fc_min=1,
    use_raw=False,
    standard_scale="var",
    #save="BE_vs_WT_matrixplot_starred.pdf",
)


In [ ]:
# Compute and store `(de_vs_rq, genes_vs_rq)`.
de_vs_rq, genes_vs_rq = make_be_vs_reference_matrixplot(
    VCMs,
    reference_group="R636Q__no_Cas9",
    target_groups=be_groups,
    groupby="Cas9_sample_group",
    n_top=10,
    p_adj_cutoff=0.05,
    log2fc_min=1,
    use_raw=False,
    standard_scale="var",
    #save="BE_vs_R636Q_matrixplot_starred.pdf",
)


In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

# Set `groupby` for the following analysis.
groupby = "Cas9_sample_group"

# Define the values used for `be_groups`.
be_groups = [
    "BE__Cas9_full",
    "BE__NtermCas9",
    "BE__CtermCas9",
    "BE__no_Cas9",
]

# Define the values used for `pretty_names`.
pretty_names = {
    "BE__Cas9_full": "BE Cas9 full",
    "BE__NtermCas9": "BE Cas9 Nterm",
    "BE__CtermCas9": "BE Cas9 Cterm",
    "BE__no_Cas9": "BE no Cas9",
}


In [ ]:
# Define `collect_filtered_markers()` for reuse in the analysis below.
def collect_filtered_markers(
    adata,
    reference_group,
    target_groups,
    groupby="Cas9_sample_group",
    method="wilcoxon",
    use_raw=False,
    min_logfc=1,
    max_padj=0.05,
):
    marker_tables = {}
    sig_gene_sets = {}

    for target in target_groups:
        sub = adata[adata.obs[groupby].isin([target, reference_group])].copy()
        key = f"DE__{target}__vs__{reference_group}"

        sc.tl.rank_genes_groups(
            sub,
            groupby=groupby,
            groups=[target],
            reference=reference_group,
            method=method,
            use_raw=use_raw,
            pts=True,
            tie_correct=True,
            key_added=key,
        )

        df = sc.get.rank_genes_groups_df(
            sub,
            group=target,
            key=key,
            pval_cutoff=max_padj,
            log2fc_min=min_logfc,
        ).copy()

        df = df.dropna(subset=["names", "logfoldchanges", "scores", "pvals_adj"])
        df = df[df["logfoldchanges"] > min_logfc]
        df = df[df["pvals_adj"] < max_padj]
        df = df.drop_duplicates(subset="names")

        marker_tables[target] = df
        sig_gene_sets[target] = set(df["names"].tolist())

    return marker_tables, sig_gene_sets


In [ ]:
# Define `choose_top_genes()` for reuse in the analysis below.
def choose_top_genes(
    marker_tables,
    n_top=5,
    sort_by="logfoldchanges",   # or "scores"
    pretty_names=None,
    unique_across_groups=False,
):
    if sort_by not in ["logfoldchanges", "scores"]:
        raise ValueError("sort_by must be 'logfoldchanges' or 'scores'")

    if pretty_names is None:
        pretty_names = {}

    gene_blocks = {}
    flat_genes = []
    seen = set()

    for target, df in marker_tables.items():
        if sort_by == "logfoldchanges":
            df2 = df.sort_values(
                ["logfoldchanges", "scores"],
                ascending=[False, False]
            )
        else:
            df2 = df.sort_values(
                ["scores", "logfoldchanges"],
                ascending=[False, False]
            )

        genes = []
        for g in df2["names"].tolist():
            if unique_across_groups and g in seen:
                continue
            genes.append(g)
            seen.add(g)
            if len(genes) == n_top:
                break

        if len(genes) > 0:
            label = pretty_names.get(target, target)
            gene_blocks[label] = genes
            flat_genes.extend(genes)

    return gene_blocks, flat_genes


In [ ]:
# Define `plot_expression_matrix_with_all_stars()` for reuse in the analysis below.
def plot_expression_matrix_with_all_stars(
    adata,
    reference_group,
    target_groups,
    gene_blocks,
    flat_genes,
    sig_gene_sets,
    groupby="Cas9_sample_group",
    use_raw=False,
    standard_scale="var",
    title=None,
    save=None,
):
    plot_order = [reference_group] + list(target_groups)
    plot_adata = adata[adata.obs[groupby].isin(plot_order)].copy()

    total_genes = len(flat_genes)
    fig_w = max(8, total_genes * 0.45)

    mp = sc.pl.matrixplot(
        plot_adata,
        var_names=gene_blocks,
        groupby=groupby,
        categories_order=plot_order,
        use_raw=use_raw,
        standard_scale=standard_scale,
        dendrogram=False,
        figsize=(fig_w, 4),
        return_fig=True,
    )

    mp.make_figure()
    axes = mp.get_axes()
    ax = axes["mainplot_ax"]

    y_labels = [t.get_text() for t in ax.get_yticklabels()]
    y_ticks = ax.get_yticks()
    y_map = dict(zip(y_labels, y_ticks))

    x_ticks = ax.get_xticks()

    # for each plotted gene column:
    # add a star in every BE row where that gene is significant
    for x, gene in zip(x_ticks, flat_genes):
        for target in target_groups:
            if gene in sig_gene_sets.get(target, set()) and target in y_map:
                ax.text(
                    x, y_map[target], "★",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                    fontweight="bold"
                )

    if title is not None:
        ax.set_title(title, pad=20)

    if save is not None:
        mp.savefig(save, bbox_inches="tight")

    plt.show()


In [ ]:
# Compute and store `(markers_vs_wt, sig_vs_wt)`.
markers_vs_wt, sig_vs_wt = collect_filtered_markers(
    VCMs,
    reference_group="WT__no_Cas9",
    target_groups=be_groups,
    groupby=groupby,
    method="wilcoxon",
    use_raw=False,
    min_logfc=1,
    max_padj=0.05,
)

# Compute and store `(genes_wt_lfc, flat_wt_lfc)`.
genes_wt_lfc, flat_wt_lfc = choose_top_genes(
    markers_vs_wt,
    n_top=10,
    sort_by="scores",
    pretty_names=pretty_names,
)

# Run `plot_expression_matrix_with_all_stars` for this analysis step.
plot_expression_matrix_with_all_stars(
    VCMs,
    reference_group="WT__no_Cas9",
    target_groups=be_groups,
    gene_blocks=genes_wt_lfc,
    flat_genes=flat_wt_lfc,
    sig_gene_sets=sig_vs_wt,
    groupby=groupby,
    use_raw=False,
    standard_scale="var",
    #title="BE subgroups vs WT — top filtered genes by logFC",
    #save="BE_vs_WT_topFiltered_byLogFC_allStars_matrixplot.pdf",
)


In [ ]:
# Compute and store `(markers_vs_rq, sig_vs_rq)`.
markers_vs_rq, sig_vs_rq = collect_filtered_markers(
    VCMs,
    reference_group="R636Q__no_Cas9",
    target_groups=be_groups,
    groupby=groupby,
    method="wilcoxon",
    use_raw=False,
    min_logfc=1,
    max_padj=0.05,
)

# Compute and store `(genes_rq_lfc, flat_rq_lfc)`.
genes_rq_lfc, flat_rq_lfc = choose_top_genes(
    markers_vs_rq,
    n_top=10,
    sort_by="scores",
    pretty_names=pretty_names,
)

# Run `plot_expression_matrix_with_all_stars` for this analysis step.
plot_expression_matrix_with_all_stars(
    VCMs,
    reference_group="R636Q__no_Cas9",
    target_groups=be_groups,
    gene_blocks=genes_rq_lfc,
    flat_genes=flat_rq_lfc,
    sig_gene_sets=sig_vs_rq,
    groupby=groupby,
    use_raw=False,
    standard_scale="var",
    #title="BE subgroups vs R636Q — top filtered genes by logFC",
    #save="BE_vs_R636Q_topFiltered_byLogFC_allStars_matrixplot.pdf",
)


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby="Cas9_sample_group", var_names = genes_wt_lfc, standard_scale="var", use_raw = False)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "Cas9_status"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "sample_type"

# work from the object as-is (uses the categorical orders you already set)
obs = VCMs.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
props_sample


In [ ]:

# ---- Plot 1: proportions per sample ----
fig, ax = plt.subplots(figsize=(5, 5))
# Run `props_sample.plot` for this analysis step.
props_sample.plot(kind="bar", stacked=True, ax=ax, color=["#6C5C8D", "#88CCEE", "#97B1AB", "lightgrey"], width=0.9)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Run `ax.set_ylabel` for this analysis step.
ax.set_ylabel("Cas9 proportion")
# Run `ax.set_xlabel` for this analysis step.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Run `ax.legend` for this analysis step.
ax.legend(title="celltype", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "Cas9status_proportions_per_sample_integratedbysamp.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Run `plt.show` for this analysis step.
plt.show()


In [ ]:
# Define the values used for `genes`.
genes = ['Rbm20', 'Myh7b', 'Myh6', 'Tnnt2', 'Atp2a2', 'Ankrd1', 'Xirp2', 'Nppa', 'Nppb', 'Hcn1',
         "F830016B08Rik", "C_terminal_Cas9", "Abe8e_with_N_terminal_Cas9", 
         'Igtp', 'Iigp1', 'Stat1', 'Il31ra', 'Rbfox1', 'Rgs6']


In [ ]:
# Plot the distribution of the selected measurement across groups.
sc.pl.violin(VCMs, groupby = "celltype", keys = "Macrod1")


In [ ]:
# Plot the distribution of the selected measurement across groups.
sc.pl.violin(VCMs, groupby = "celltype", keys = "Macrod1")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = 'celltype', standard_scale="var", var_names= genes, dendrogram=False, categories_order=order)


In [ ]:
# Plot the selected expression matrix across groups.
sc.pl.matrixplot(
    VCMs,
    var_names= genes,
    groupby="celltype",
    standard_scale="var"
)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "sample_type"

# work from the object as-is (uses the categorical orders you already set)
obs = VCMs.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:

# ---- Plot 1: proportions per sample ----
fig, ax = plt.subplots(figsize=(5, 5))
# Run `props_sample.plot` for this analysis step.
props_sample.plot(kind="bar", stacked=True, ax=ax, color=set1_9, width=0.9)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Run `ax.set_ylabel` for this analysis step.
ax.set_ylabel("Cell type proportion")
# Run `ax.set_xlabel` for this analysis step.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Run `ax.legend` for this analysis step.
ax.legend(title="celltype", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()
#fig.savefig(os.path.join(out_dir, "celltype_proportions_per_sample.pdf"),
            #dpi=300, bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
# ---- Plot 2: mean proportions per group ----
fig, ax = plt.subplots(figsize=(4.5, 5))
# Run `group_means.plot` for this analysis step.
group_means.plot(kind="bar", stacked=True, ax=ax, color=set1_9, width=0.8)
# Run `ax.grid` for this analysis step.
ax.grid(False)
# Run `ax.set_ylabel` for this analysis step.
ax.set_ylabel("Mean cell type proportion")
# Run `ax.set_xlabel` for this analysis step.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Run `ax.legend` for this analysis step.
ax.legend(title="celltype", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
# Run `plt.tight_layout` for this analysis step.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_mean_proportions_per_group_integratedbysample.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Run `plt.show` for this analysis step.
plt.show()
